# RSNA 2022 Spine Fracture Detection – Phase 2
## Model Building, Evaluation & Writing

This notebook implements Phase 2 of the project: training and evaluating ML classifiers to predict the visibility of cervical vertebrae **C1 through C7** in CT scan slices using metadata features.

### Phase 2 Workflow

1. **Preprocessing** — Clean data, encode features, scale, split, handle class imbalance
2. **Model Implementation** — Train Logistic Regression, Random Forest, and XGBoost for each of the 7 vertebrae
3. **Ablation Study** — Hyperparameter tuning via Grid Search with Cross-Validation
4. **Evaluation & Comparison** — Compare all models with classification metrics, confusion matrices, and ROC-AUC

### Design Decisions (Justified)

- **Problem formulation**: Multi-label binary classification. Each slice has 7 independent binary labels (C1–C7). We use the **Binary Relevance** approach: train one model per vertebra per algorithm (7 vertebrae × 3 algorithms = 21 models total).
- **Sampling**: A stratified subsample of ~150,000 slices is used (split by `StudyInstanceUID` to prevent patient-level data leakage). This makes hyperparameter tuning feasible without compromising statistical power.
- **Splitting**: Train/test split is performed at the **study level**, not the row level. All slices from a given patient go to either train or test, never both. Essential for medical data to obtain honest performance estimates.
- **Models**: Logistic Regression (linear baseline), Random Forest (tree-based bagging), XGBoost (gradient boosting). These cover three distinct algorithm families.
- **Imbalance handling**: `class_weight='balanced'` for LogReg/RF and per-vertebra `scale_pos_weight` for XGBoost. Each vertebra has its own imbalance ratio.
- **Hyperparameter tuning**: Performed on a representative target (C1) and the optimal configuration is applied across all 7 vertebrae. This is a common, defensible practice in multi-label problems to balance rigor with computational feasibility.

# Part 1: Preprocessing

## 1.1 Imports and Setup

In [4]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing tools from scikit-learn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# Reproducibility: fix the random seed so results are repeatable across runs.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print('Libraries imported successfully.')

Libraries imported successfully.


## 1.2 Load the Dataset

We reuse the same `meta_train_with_vertebrae.csv` file from Phase 1.
Make sure the file is at `rsna_metadata/meta_train_with_vertebrae.csv` (i.e. you've already run the download/unzip cells from Phase 1).

In [5]:
df = pd.read_csv('rsna_metadata/meta_train_with_vertebrae.csv')

print(f'Loaded dataset with shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Loaded dataset with shape: (711601, 16)
Columns: ['StudyInstanceUID', 'Slice', 'ImageHeight', 'ImageWidth', 'SliceThickness', 'ImagePositionPatient_x', 'ImagePositionPatient_y', 'ImagePositionPatient_z', 'SliceRatio', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']


## 1.3 Data Cleaning Recap

From Phase 1, we know:
- No missing values
- No duplicate rows
- Identifiers (StudyInstanceUID) are consistent

We re-verify quickly here for completeness.

In [6]:
print('Total missing values:', df.isnull().sum().sum())
print('Duplicate rows:      ', df.duplicated().sum())
print('Unique studies:      ', df['StudyInstanceUID'].nunique())

Total missing values: 0
Duplicate rows:       0
Unique studies:       2019


## 1.4 Define Targets and Features

We're doing **multi-label classification** with 7 binary targets (C1–C7).

Important: when predicting any vertebra, the **other vertebrae are NOT used as features.** In real-world deployment, you wouldn't know any of them ahead of time — they all come from the same image. Using them as features would constitute target leakage. We exclude all 7 vertebra columns from the feature set; they live exclusively in `Y`.

Other drops:
1. **`StudyInstanceUID`** — unique identifier, no predictive value. We **keep it temporarily** for the group-aware split, then drop it before modeling.
2. **`ImageHeight`** — Phase 1 showed r ≈ 1.00 with `ImageWidth` (they're always equal). Keeping both adds no information and creates multicollinearity for linear models.

In [7]:
# Define our 7 target columns
TARGETS = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']
REDUNDANT_FEATURES = ['ImageHeight']  # r ≈ 1.0 with ImageWidth

# Features = everything except targets, identifier, and redundant columns
feature_cols = [
    col for col in df.columns
    if col not in TARGETS + ['StudyInstanceUID'] + REDUNDANT_FEATURES
]

print(f'Targets ({len(TARGETS)}):  {TARGETS}')
print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'Dropped (redundant):       {REDUNDANT_FEATURES}')
print(f'Dropped (identifier):      ["StudyInstanceUID"] (kept for group-aware splitting)')

Targets (7):  ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']
Features (7): ['Slice', 'ImageWidth', 'SliceThickness', 'ImagePositionPatient_x', 'ImagePositionPatient_y', 'ImagePositionPatient_z', 'SliceRatio']
Dropped (redundant):       ['ImageHeight']
Dropped (identifier):      ["StudyInstanceUID"] (kept for group-aware splitting)


## 1.5 Stratified Subsampling by Study

The full dataset has 711,601 slices, which makes hyperparameter tuning slow. We subsample ~150,000 slices by sampling **whole studies** (not random rows). This preserves each patient's data as an intact unit.

Why whole studies?
- Slices within a study are sequential and highly correlated (consecutive slice numbers, same patient anatomy)
- Random row sampling would split patients mid-study, which complicates downstream group-aware splitting
- Sampling whole studies preserves the natural unit of medical data: the patient

In [23]:
# Use the full dataset (no subsampling)
df_sampled = df.copy()

print(f'Using full dataset, shape: {df_sampled.shape}')
print(f'Total studies: {df_sampled["StudyInstanceUID"].nunique()}')

# Verify class distribution across all 7 targets
print('\nPositive-class proportion per vertebra:')
print(df_sampled[TARGETS].mean().round(4))

Using full dataset, shape: (711601, 16)
Total studies: 2019

Positive-class proportion per vertebra:
C1    0.1241
C2    0.2200
C3    0.1212
C4    0.1315
C5    0.1370
C6    0.1344
C7    0.1502
dtype: float64


In [ ]:
"""# Get list of unique study IDs and compute average slices per study
all_studies = df['StudyInstanceUID'].unique()
avg_slices_per_study = len(df) / len(all_studies)

print(f'Total studies: {len(all_studies)}')
print(f'Average slices per study: {avg_slices_per_study:.1f}')

TARGET_SAMPLE_SIZE = 150_000
n_studies_to_sample = int(np.ceil(TARGET_SAMPLE_SIZE / avg_slices_per_study))
print(f'Studies needed for ~{TARGET_SAMPLE_SIZE:,} slices: {n_studies_to_sample}')

rng = np.random.default_rng(RANDOM_SEED)
sampled_studies = rng.choice(all_studies, size=n_studies_to_sample, replace=False)
df_sampled = df[df['StudyInstanceUID'].isin(sampled_studies)].copy()

print(f'\nSampled dataset shape: {df_sampled.shape}')
print(f'Sampled studies: {df_sampled["StudyInstanceUID"].nunique()}')

# Verify class distribution is preserved across all 7 targets
print('\nPositive-class proportion per vertebra (sample vs full):')
compare_df = pd.DataFrame({
    'full_dataset': df[TARGETS].mean(),
    'sample':       df_sampled[TARGETS].mean()
})
print(compare_df.round(4))"""

## 1.6 Train/Test Split — by Study (Group-Aware)

We split the **studies** (not the rows). This guarantees no patient appears in both train and test.

**`GroupShuffleSplit`** respects group boundaries. Ratio: **80% train / 20% test**.

In [24]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)

train_idx, test_idx = next(gss.split(
    X=df_sampled,
    y=df_sampled[TARGETS[0]],  # any target works; splitting uses groups, not y
    groups=df_sampled['StudyInstanceUID']
))

df_train = df_sampled.iloc[train_idx].copy()
df_test  = df_sampled.iloc[test_idx].copy()

# Sanity check: no overlap of studies
train_studies = set(df_train['StudyInstanceUID'].unique())
test_studies  = set(df_test['StudyInstanceUID'].unique())
overlap = train_studies & test_studies

print(f'Training set: {len(df_train):,} slices from {len(train_studies)} studies')
print(f'Test set:     {len(df_test):,} slices from {len(test_studies)} studies')
print(f'Overlapping studies (should be 0): {len(overlap)}')

print('\nPositive-class proportion per vertebra (train vs test):')
split_compare = pd.DataFrame({
    'train': df_train[TARGETS].mean(),
    'test':  df_test[TARGETS].mean()
})
print(split_compare.round(4))

Training set: 568,399 slices from 1615 studies
Test set:     143,202 slices from 404 studies
Overlapping studies (should be 0): 0

Positive-class proportion per vertebra (train vs test):
     train    test
C1  0.1228  0.1290
C2  0.2199  0.2201
C3  0.1212  0.1209
C4  0.1316  0.1311
C5  0.1368  0.1377
C6  0.1342  0.1349
C7  0.1506  0.1486


## 1.7 Separate Features (X) and Targets (Y)

Standard sklearn convention: `X` is the feature matrix. For multi-label, `Y` is a DataFrame with one column per target.

Now that the split is done, we drop `StudyInstanceUID` since we no longer need it.

In [25]:
X_train = df_train[feature_cols]
Y_train = df_train[TARGETS]

X_test = df_test[feature_cols]
Y_test = df_test[TARGETS]

print('Feature matrix shapes:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')
print()
print('Target matrix shapes:')
print(f'  Y_train: {Y_train.shape}  (rows × 7 targets)')
print(f'  Y_test:  {Y_test.shape}')

Feature matrix shapes:
  X_train: (568399, 7)
  X_test:  (143202, 7)

Target matrix shapes:
  Y_train: (568399, 7)  (rows × 7 targets)
  Y_test:  (143202, 7)


## 1.8 Feature Scaling

**Why scale?** Features have very different scales: `Slice` (1–1082), `SliceThickness` (0.5–1.0), `ImagePositionPatient_z` (-1687 to +2222), `SliceRatio` (0–1).

**Logistic Regression** is sensitive to scale (gradient optimization, regularization unfair to large-magnitude features).

**Tree-based models** (Random Forest, XGBoost) are scale-invariant — they compare values to thresholds, not compute distances.

We keep both versions: scaled for LogReg, original for tree models.

**Critical**: fit scaler on `X_train` only, then transform both. Never fit on test data.

In [26]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)  # learns mean/std AND transforms
X_test_scaled  = scaler.transform(X_test)       # applies same mean/std (no re-fitting!)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_cols, index=X_test.index)

print('Training set after scaling (first 5 features):')
print(X_train_scaled.describe().loc[['mean', 'std']].iloc[:, :5].round(4))
print()
print('Test set after scaling (first 5 features):')
print('Note: test mean/std may differ slightly from 0/1 — this is correct.')
print('We applied the training set\'s statistics, not the test set\'s own.')
print(X_test_scaled.describe().loc[['mean', 'std']].iloc[:, :5].round(4))

Training set after scaling (first 5 features):
      Slice  ImageWidth  SliceThickness  ImagePositionPatient_x  \
mean   -0.0        -0.0             0.0                    -0.0   
std     1.0         1.0             1.0                     1.0   

      ImagePositionPatient_y  
mean                    -0.0  
std                      1.0  

Test set after scaling (first 5 features):
Note: test mean/std may differ slightly from 0/1 — this is correct.
We applied the training set's statistics, not the test set's own.
       Slice  ImageWidth  SliceThickness  ImagePositionPatient_x  \
mean  0.0110      0.0801         -0.0346                 -0.0943   
std   1.0034      2.0205          0.9940                  1.0204   

      ImagePositionPatient_y  
mean                 -0.0103  
std                   0.9824  


## 1.9 Class Imbalance — Per-Vertebra Strategy

Each of the 7 vertebrae has its own imbalance ratio. Phase 1 showed C1 ≈ 12% positive, C2 ≈ 22%, others ≈ 12–15%. We compute a separate `scale_pos_weight` per vertebra for XGBoost.

**Why imbalance matters:**
- A model that always predicts 0 would achieve 78–88% accuracy — useless but high-scoring on the wrong metric
- Standard accuracy is misleading; we'll prioritize **precision, recall, F1, and ROC-AUC** instead

**Approach:**
- **LogReg / RF**: `class_weight='balanced'` (sklearn auto-handles, same syntax for every vertebra)
- **XGBoost**: `scale_pos_weight = (# negatives) / (# positives)` per vertebra

In [27]:
scale_pos_weights = {}

print(f"{'Vertebra':<10} {'# Negative':>12} {'# Positive':>12} {'Pos %':>8} {'scale_pos_weight':>18}")
print('-' * 62)
for target in TARGETS:
    n_neg = (Y_train[target] == 0).sum()
    n_pos = (Y_train[target] == 1).sum()
    ratio = n_neg / n_pos
    scale_pos_weights[target] = ratio
    pct = n_pos / (n_neg + n_pos) * 100
    print(f'{target:<10} {n_neg:>12,} {n_pos:>12,} {pct:>7.2f}% {ratio:>18.3f}')

print(f'\nscale_pos_weights dict ready for XGBoost: {scale_pos_weights}')

Vertebra     # Negative   # Positive    Pos %   scale_pos_weight
--------------------------------------------------------------
C1              498,595       69,804   12.28%              7.143
C2              443,385      125,014   21.99%              3.547
C3              499,489       68,910   12.12%              7.248
C4              493,619       74,780   13.16%              6.601
C5              490,614       77,785   13.68%              6.307
C6              492,098       76,301   13.42%              6.449
C7              482,773       85,626   15.06%              5.638

scale_pos_weights dict ready for XGBoost: {'C1': np.float64(7.142785513724142), 'C2': np.float64(3.5466827715295888), 'C3': np.float64(7.248425482513423), 'C4': np.float64(6.60094945172506), 'C5': np.float64(6.307308607057916), 'C6': np.float64(6.449430544815926), 'C7': np.float64(5.638158970406185)}


## 1.10 Preprocessing Summary

Ready for modeling:

| Object | Shape | Purpose |
|--------|-------|---------|
| `X_train` | (~120k, 9) | Original-scale features for tree models |
| `X_train_scaled` | (~120k, 9) | StandardScaled features for Logistic Regression |
| `Y_train` | (~120k, 7) | Training targets for all 7 vertebrae |
| `X_test` | (~30k, 9) | Test features (original scale) |
| `X_test_scaled` | (~30k, 9) | Test features (scaled) |
| `Y_test` | (~30k, 7) | Test targets |
| `scale_pos_weights` | dict (7 items) | Per-vertebra imbalance weight for XGBoost |

Preprocessing decisions documented:
1. ✅ No missing values to handle
2. ✅ No duplicates to remove
3. ✅ Identifier column dropped after splitting (StudyInstanceUID)
4. ✅ Redundant feature dropped (ImageHeight ≈ ImageWidth)
5. ✅ Subsampled to ~150k rows preserving study integrity
6. ✅ Train/test split at the study level (no patient leakage)
7. ✅ Feature scaling fitted on train only, applied to both
8. ✅ Per-vertebra class imbalance strategy defined

# Part 2: Model Implementation

We train **4 algorithms × 7 vertebrae = 28 models total** using the Binary Relevance approach (one model per target).

For each model, we save:
- The fitted model object (so we can reuse it later without retraining)
- The hard predictions on the test set (used for accuracy, F1, etc.)
- The predicted probabilities on the test set (used for ROC-AUC)

We use **default-ish hyperparameters** here as a baseline. Tuning comes in Part 3 (Ablation Study).

## 2.1 Imports for Modeling

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import time  # to time how long each model takes to train

## 2.2 Storage Dictionaries

We use nested dictionaries to store everything: `models[algorithm][vertebra]` gives the trained model, and similarly for predictions and probabilities.

In [28]:
# Storage for trained models and their outputs
models = {
    'LogisticRegression': {},
    'DecisionTree': {},
    'RandomForest': {},
    'XGBoost': {}
}

predictions = {
    'LogisticRegression': {},
    'DecisionTree': {},
    'RandomForest': {},
    'XGBoost': {}
}

probabilities = {
    'LogisticRegression': {},
    'DecisionTree': {},
    'RandomForest': {},
    'XGBoost': {}
}

print('Storage initialized.')

Storage initialized.


## 2.3 Train Logistic Regression — One per Vertebra

**Logistic Regression** is our linear baseline. Despite the name, it's a *classification* algorithm. It learns a weighted combination of the features and passes it through a sigmoid function to output a probability.

Key arguments:
- `class_weight='balanced'` — automatically up-weights the minority class
- `max_iter=1000` — allow enough iterations for the optimizer to converge
- `random_state=RANDOM_SEED` — for reproducibility

We use the **scaled features** (`X_train_scaled`) because LogReg is sensitive to feature scales.

In [29]:
print('Training Logistic Regression models...')
print('-' * 50)

for vertebra in TARGETS:
    start = time.time()
    
    # Create and train the model
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_SEED
    )
    model.fit(X_train_scaled, Y_train[vertebra])
    
    # Make predictions and get probabilities on the test set
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]  # probability of class 1
    
    # Save everything
    models['LogisticRegression'][vertebra]        = model
    predictions['LogisticRegression'][vertebra]   = y_pred
    probabilities['LogisticRegression'][vertebra] = y_proba
    
    elapsed = time.time() - start
    print(f'  {vertebra}: done in {elapsed:.1f}s')

print('\nAll 7 Logistic Regression models trained.')

Training Logistic Regression models...
--------------------------------------------------
  C1: done in 0.3s
  C2: done in 0.2s
  C3: done in 0.2s
  C4: done in 0.2s
  C5: done in 0.2s
  C6: done in 0.2s
  C7: done in 0.2s

All 7 Logistic Regression models trained.


## 2.4 Train Decision Tree — One per Vertebra

**Decision Tree** classifies data by asking a sequence of yes/no questions about feature values, branching down a tree until it reaches a leaf node that gives the prediction. For example, the model might learn rules like:

> If `SliceRatio > 0.7` and `ImagePositionPatient_z < -300` → predict C1 = 0

It builds the tree by repeatedly choosing the feature and threshold that best separates the classes at each node.

Key arguments:
- `class_weight='balanced'` — handles class imbalance (same as LogReg/RF)
- `max_depth=20` — limits how deep the tree can grow, preventing severe overfitting on 568k rows
- `random_state=RANDOM_SEED` — for reproducibility

We use the **unscaled features** — Decision Trees don't need scaling because they only compare values to thresholds, not compute distances.

**Note**: A single Decision Tree usually performs worse than the Random Forest (which is just many trees averaged). This comparison is informative for the paper.

In [30]:
from sklearn.tree import DecisionTreeClassifier

print('Training Decision Tree models...')
print('-' * 50)

for vertebra in TARGETS:
    start = time.time()
    
    model = DecisionTreeClassifier(
        class_weight='balanced',
        max_depth=20,
        random_state=RANDOM_SEED
    )
    model.fit(X_train, Y_train[vertebra])
    
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    models['DecisionTree'][vertebra]        = model
    predictions['DecisionTree'][vertebra]   = y_pred
    probabilities['DecisionTree'][vertebra] = y_proba
    
    elapsed = time.time() - start
    print(f'  {vertebra}: done in {elapsed:.1f}s')

print('\nAll 7 Decision Tree models trained.')

Training Decision Tree models...
--------------------------------------------------
  C1: done in 1.8s
  C2: done in 1.9s
  C3: done in 1.7s
  C4: done in 1.6s
  C5: done in 1.7s
  C6: done in 1.5s
  C7: done in 1.6s

All 7 Decision Tree models trained.


## 2.5 Train Random Forest — One per Vertebra

**Random Forest** is an ensemble of decision trees. Each tree is trained on a random subset of the data and features; predictions are averaged across trees. This makes it robust and less prone to overfitting than a single tree.

Key arguments:
- `n_estimators=100` — number of trees in the forest (100 is a sensible default)
- `class_weight='balanced'` — handles imbalance
- `n_jobs=-1` — use all CPU cores (trees train independently, so this parallelizes well)
- `random_state=RANDOM_SEED` — for reproducibility

We use the **unscaled features** (`X_train`) — tree-based models don't need scaling.

**Note**: This is the slowest of the three. Expect a few minutes total across all 7 vertebrae.

In [31]:
print('Training Random Forest models...')
print('-' * 50)

for vertebra in TARGETS:
    start = time.time()
    
    model = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    model.fit(X_train, Y_train[vertebra])
    
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    models['RandomForest'][vertebra]        = model
    predictions['RandomForest'][vertebra]   = y_pred
    probabilities['RandomForest'][vertebra] = y_proba
    
    elapsed = time.time() - start
    print(f'  {vertebra}: done in {elapsed:.1f}s')

print('\nAll 7 Random Forest models trained.')

Training Random Forest models...
--------------------------------------------------
  C1: done in 6.8s
  C2: done in 7.0s
  C3: done in 6.6s
  C4: done in 6.9s
  C5: done in 6.9s
  C6: done in 6.4s
  C7: done in 7.3s

All 7 Random Forest models trained.


## 2.6 Train XGBoost — One per Vertebra

**XGBoost** is a gradient boosting algorithm — it builds trees sequentially, where each new tree corrects the errors of the previous ones. It's known for high performance on tabular data.

Key arguments:
- `n_estimators=100` — number of boosting rounds
- `scale_pos_weight=...` — handles imbalance (different for each vertebra, computed in Part 1)
- `eval_metric='logloss'` — avoids a deprecation warning by setting the metric explicitly
- `n_jobs=-1` — use all CPU cores
- `random_state=RANDOM_SEED` — for reproducibility

We use the **unscaled features** — XGBoost doesn't need scaling either.

In [32]:
print('Training XGBoost models...')
print('-' * 50)

for vertebra in TARGETS:
    start = time.time()
    
    model = XGBClassifier(
        n_estimators=100,
        scale_pos_weight=scale_pos_weights[vertebra],
        eval_metric='logloss',
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    model.fit(X_train, Y_train[vertebra])
    
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    models['XGBoost'][vertebra]        = model
    predictions['XGBoost'][vertebra]   = y_pred
    probabilities['XGBoost'][vertebra] = y_proba
    
    elapsed = time.time() - start
    print(f'  {vertebra}: done in {elapsed:.1f}s')

print('\nAll 7 XGBoost models trained.')

Training XGBoost models...
--------------------------------------------------
  C1: done in 1.0s
  C2: done in 0.9s
  C3: done in 0.9s
  C4: done in 0.9s
  C5: done in 0.9s
  C6: done in 0.9s
  C7: done in 1.0s

All 7 XGBoost models trained.


## 2.7 Quick Check

Let's verify all 28 models are stored correctly and look at quick baseline accuracy. We'll do the full evaluation (precision, recall, F1, ROC-AUC, confusion matrices) in Part 4 — this is just a quick check that nothing is obviously broken.

In [33]:
from sklearn.metrics import accuracy_score

# Build a quick accuracy summary table
summary_rows = []
for algo in ['LogisticRegression', 'RandomForest', 'XGBoost', 'DecisionTree']:
    row = {'algorithm': algo}
    for vertebra in TARGETS:
        acc = accuracy_score(Y_test[vertebra], predictions[algo][vertebra])
        row[vertebra] = round(acc, 3)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('algorithm')
print('Test set accuracy (baseline models, no hyperparameter tuning):')
print(summary_df)
print('\nNote: Accuracy alone is misleading on imbalanced data. We will report')
print('precision, recall, F1, and ROC-AUC in Part 4 for a meaningful comparison.')

Test set accuracy (baseline models, no hyperparameter tuning):
                       C1     C2     C3     C4     C5     C6     C7
algorithm                                                          
LogisticRegression  0.776  0.777  0.590  0.510  0.599  0.688  0.767
RandomForest        0.977  0.982  0.979  0.976  0.978  0.974  0.971
XGBoost             0.969  0.979  0.975  0.974  0.973  0.971  0.969
DecisionTree        0.967  0.975  0.974  0.969  0.970  0.969  0.968

Note: Accuracy alone is misleading on imbalanced data. We will report
precision, recall, F1, and ROC-AUC in Part 4 for a meaningful comparison.


# Part 3: Ablation Study (Hyperparameter Tuning)

We tune each algorithm's hyperparameters using **Grid Search with 5-fold Cross-Validation**, scoring on **F1** (more meaningful than accuracy for our imbalanced classes).

To keep runtime manageable while staying rigorous:
- Tuning is performed on **C1 only** (representative target)
- The full 568k training set is used (no subsampling for tuning)
- Grid sizes are kept compact: 3–4 values per hyperparameter, focused on impactful ones

The optimal hyperparameters found for C1 are then applied to refit all 7 vertebrae per algorithm. This is a standard practice in multi-label problems and will be documented in the paper.

## 3.1 Initial Setup

In [35]:
from sklearn.model_selection import GridSearchCV

# Storage for the tuning results
best_params = {}      # best hyperparameter combo per algorithm
cv_results  = {}      # full CV results table per algorithm

# Use C1 only for tuning
y_tune = Y_train['C1']

print('Setup complete. Tuning will use C1 as the representative target.')
print(f'Tuning data shape: X={X_train.shape}, y={y_tune.shape}')

Setup complete. Tuning will use C1 as the representative target.
Tuning data shape: X=(568399, 7), y=(568399,)


## 3.2 Tune Logistic Regression

Logistic Regression has one main hyperparameter worth tuning: **`C`** (regularization strength). Smaller C = more regularization = simpler model.

We also explore the **`penalty`** type:
- `'l2'` (Ridge) — shrinks coefficients toward zero
- `'l1'` (Lasso) — can drive some coefficients to exactly zero, performing implicit feature selection

Grid: 4 values of C × 2 penalties = **8 combinations** × 5 CV folds = 40 fits total.

In [36]:
# Define the grid
logreg_grid = {
    'C':       [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2']
}

# Base estimator with fixed settings (same as Part 2 baseline)
logreg_base = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='liblinear',          # required for l1 penalty
    random_state=RANDOM_SEED
)

# Grid search
logreg_search = GridSearchCV(
    estimator=logreg_base,
    param_grid=logreg_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

print('Starting Logistic Regression grid search...')
start = time.time()
logreg_search.fit(X_train_scaled, y_tune)
elapsed = time.time() - start
print(f'\nDone in {elapsed:.1f}s')

# Store results
best_params['LogisticRegression'] = logreg_search.best_params_
cv_results['LogisticRegression']  = pd.DataFrame(logreg_search.cv_results_)

print(f'\nBest parameters: {logreg_search.best_params_}')
print(f'Best CV F1 score: {logreg_search.best_score_:.4f}')

Starting Logistic Regression grid search...
Fitting 5 folds for each of 8 candidates, totalling 40 fits


c:\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



Done in 52.5s

Best parameters: {'C': 1.0, 'penalty': 'l1'}
Best CV F1 score: 0.5149


## 3.3 Tune Decision Tree

Decision Trees are very sensitive to two hyperparameters that control overfitting:

- **`max_depth`** — maximum number of question-asking levels in the tree. Too shallow = underfits (model too simple). Too deep = overfits (memorizes training data).
- **`min_samples_leaf`** — minimum number of training samples required to form a leaf node. Higher values force the tree to make broader generalizations rather than carving out tiny regions for outliers.

Grid: 4 depths × 3 leaf sizes = **12 combinations** × 5 CV folds = 60 fits total.

In [ ]:
# Define the grid
dt_grid = {
    'max_depth':        [10, 15, 20, 25, 30, 35],
    'min_samples_leaf': [1, 10, 50]
}

# Base estimator
dt_base = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=RANDOM_SEED
)

# Grid search
dt_search = GridSearchCV(
    estimator=dt_base,
    param_grid=dt_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

print('Starting Decision Tree grid search...')
start = time.time()
dt_search.fit(X_train, y_tune)
elapsed = time.time() - start
print(f'\nDone in {elapsed:.1f}s')

# Store results
best_params['DecisionTree'] = dt_search.best_params_
cv_results['DecisionTree']  = pd.DataFrame(dt_search.cv_results_)

print(f'\nBest parameters: {dt_search.best_params_}')
print(f'Best CV F1 score: {dt_search.best_score_:.4f}')

Starting Decision Tree grid search...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Done in 32.5s

Best parameters: {'max_depth': 25, 'min_samples_leaf': 1}
Best CV F1 score: 0.8726


## 3.4 Tune Random Forest

Random Forest has more hyperparameters than the others. We focus on the three with the largest impact:

- **`n_estimators`** — number of trees in the forest. More trees = more averaging, but linear runtime cost.
- **`max_depth`** — same meaning as Decision Tree. Controls tree complexity.
- **`min_samples_leaf`** — same meaning as Decision Tree. Controls leaf granularity.

To keep runtime manageable, we use a focused grid: 2 × 3 × 2 = **12 combinations** × 5 folds = 60 fits total.

Each Random Forest fit on 568k rows takes ~7s, so total runtime is estimated at 7–15 minutes.

In [39]:
# Define the grid — kept compact for runtime
rf_grid = {
    'n_estimators':     [100, 200],
    'max_depth':        [15, 25, None],
    'min_samples_leaf': [1, 10]
}

# Base estimator
rf_base = RandomForestClassifier(
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED
)

# Grid search
# Note: cv folds also use n_jobs internally, so we set GridSearchCV's n_jobs=1
# to avoid nested parallelism conflicts with rf_base's n_jobs=-1
rf_search = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_grid,
    scoring='f1',
    cv=5,
    n_jobs=1,
    verbose=2
)

print('Starting Random Forest grid search...')
print('This will take several minutes. Each line below is one fit completing.')
start = time.time()
rf_search.fit(X_train, y_tune)
elapsed = time.time() - start
print(f'\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)')

# Store results
best_params['RandomForest'] = rf_search.best_params_
cv_results['RandomForest']  = pd.DataFrame(rf_search.cv_results_)

print(f'\nBest parameters: {rf_search.best_params_}')
print(f'Best CV F1 score: {rf_search.best_score_:.4f}')

Starting Random Forest grid search...
This will take several minutes. Each line below is one fit completing.
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=100; total time=   5.2s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=100; total time=   5.1s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=100; total time=   5.0s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=100; total time=   5.4s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=100; total time=   5.0s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=200; total time=   9.9s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=200; total time=  10.0s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=200; total time=   9.9s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=200; total time=  10.1s
[CV] END .max_depth=15, min_samples_leaf=1, n_estimators=200; total time=  10.0s
[CV] END max_depth=1

## 3.5 Tune XGBoost

XGBoost has many hyperparameters. We tune the three most impactful for tabular data:

- **`n_estimators`** — number of boosting rounds (sequential trees).
- **`max_depth`** — tree depth. Note: XGBoost trees are usually shallower than Random Forest trees because each tree only needs to correct residuals.
- **`learning_rate`** — how much each new tree contributes to the ensemble. Smaller = slower but often better learning; needs more trees to compensate.

Grid: 2 × 3 × 2 = **12 combinations** × 5 folds = 60 fits.

XGBoost is fast (~1s per fit in Part 2), so expected runtime is only 1–3 minutes.

In [40]:
# Define the grid
xgb_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 6, 10],
    'learning_rate': [0.05, 0.1]
}

# Base estimator — note scale_pos_weight is for C1 since we're tuning on C1
xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weights['C1'],
    eval_metric='logloss',
    n_jobs=-1,
    random_state=RANDOM_SEED
)

# Grid search
xgb_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_grid,
    scoring='f1',
    cv=5,
    n_jobs=1,
    verbose=2
)

print('Starting XGBoost grid search...')
start = time.time()
xgb_search.fit(X_train, y_tune)
elapsed = time.time() - start
print(f'\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)')

# Store results
best_params['XGBoost'] = xgb_search.best_params_
cv_results['XGBoost']  = pd.DataFrame(xgb_search.cv_results_)

print(f'\nBest parameters: {xgb_search.best_params_}')
print(f'Best CV F1 score: {xgb_search.best_score_:.4f}')

Starting XGBoost grid search...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=100; total time=   0.6s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=100; total time=   0.5s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=100; total time=   0.5s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=100; total time=   0.5s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=100; total time=   0.5s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=200; total time=   1.0s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=200; total time=   1.0s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=200; total time=   1.0s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=200; total time=   1.0s
[CV] END ..learning_rate=0.05, max_depth=3, n_estimators=200; total time=   1.0s
[CV] END ..learning_rate=0.05, max_depth=6, n_estimators=100; total time=   0.7s
[CV] END ..learn

## 3.6 Refit All 7 Vertebrae with Optimal Hyperparameters

We now apply each algorithm's best hyperparameters (found by tuning on C1) to refit all 7 vertebrae. This overwrites the baseline Part 2 models with tuned versions.

This is the **Binary Relevance** strategy applied to multi-label classification: same hyperparameters, separate model per label.

In [ ]:
print('Refitting all algorithms × all vertebrae with tuned hyperparameters...\n')

# ----------------------------------------
# Logistic Regression
# ----------------------------------------
print('Logistic Regression:')
for vertebra in TARGETS:
    start = time.time()
    model = LogisticRegression(
        **best_params['LogisticRegression'],
        class_weight='balanced',
        max_iter=1000,
        solver='liblinear',
        random_state=RANDOM_SEED
    )
    model.fit(X_train_scaled, Y_train[vertebra])
    models['LogisticRegression'][vertebra]        = model
    predictions['LogisticRegression'][vertebra]   = model.predict(X_test_scaled)
    probabilities['LogisticRegression'][vertebra] = model.predict_proba(X_test_scaled)[:, 1]
    print(f'  {vertebra}: {time.time()-start:.1f}s')

# ----------------------------------------
# Decision Tree
# ----------------------------------------
print('\nDecision Tree:')
for vertebra in TARGETS:
    start = time.time()
    model = DecisionTreeClassifier(
        **best_params['DecisionTree'],
        class_weight='balanced',
        random_state=RANDOM_SEED
    )
    model.fit(X_train, Y_train[vertebra])
    models['DecisionTree'][vertebra]        = model
    predictions['DecisionTree'][vertebra]   = model.predict(X_test)
    probabilities['DecisionTree'][vertebra] = model.predict_proba(X_test)[:, 1]
    print(f'  {vertebra}: {time.time()-start:.1f}s')

# ----------------------------------------
# Random Forest
# ----------------------------------------
print('\nRandom Forest:')
for vertebra in TARGETS:
    start = time.time()
    model = RandomForestClassifier(
        **best_params['RandomForest'],
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    model.fit(X_train, Y_train[vertebra])
    models['RandomForest'][vertebra]        = model
    predictions['RandomForest'][vertebra]   = model.predict(X_test)
    probabilities['RandomForest'][vertebra] = model.predict_proba(X_test)[:, 1]
    print(f'  {vertebra}: {time.time()-start:.1f}s')

# ----------------------------------------
# XGBoost (uses per-vertebra scale_pos_weight)
# ----------------------------------------
print('\nXGBoost:')
for vertebra in TARGETS:
    start = time.time()
    model = XGBClassifier(
        **best_params['XGBoost'],
        scale_pos_weight=scale_pos_weights[vertebra],
        eval_metric='logloss',
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    model.fit(X_train, Y_train[vertebra])
    models['XGBoost'][vertebra]        = model
    predictions['XGBoost'][vertebra]   = model.predict(X_test)
    probabilities['XGBoost'][vertebra] = model.predict_proba(X_test)[:, 1]
    print(f'  {vertebra}: {time.time()-start:.1f}s')

print('\nAll 28 models refit with tuned hyperparameters.')